# Ring-Light Best Stack — Forehead Lab Inference (Colab)

Production-style **chart-free** forehead Lab from a Variable Lighting **torch zip** (no-flash + flash DNGs + Apple landmarks). ROI matches the **FitSkin forehead** scan site (Booth Lighting reference).

## Color stack (pinned on n=84 ring-light eval)

| Step | What |
|---|---|
| 1 | Pre-AWB demosaic → reflectance \(R_0=\sqrt{A_0\odot B_0'}\) |
| 2 | Apple Vision **forehead** mask |
| 3 | **`tier3_affine`** indoor RGB→XYZ |
| 4 | **`hybrid_deploy` CAT** — Lu+torch SPD on F12/warm; frozen 5500 K on D65 |
| 5 | **Illuminant-routed multi-Lab corrector** (`W_d65` / `W_f12`) |
| 6 | FairFace-7 ethnicity → specular/shadow sampling on forehead → Lab |

Pinned cohort mean ΔE₀₀ (legacy cheek ROI vs forehead FitSkin): **~8.5** with hybrid_deploy + multi-lab. Re-run ring eval with `--roi forehead` after pulling this update for apples-to-apples numbers.

## What each cell does

| Cell | Purpose |
|---|---|
| **1 — Setup** | Install deps, clone/pull GitHub repo, download FairFace-7 weights, load calibration paths |
| **2a — Upload** | *(Optional)* Pick one `*Torch.zip` from your computer if not using repo demos |
| **1b — Torch SPD** | *(Run after Cell 1)* Plot torch flash spectrum + RGB prior |
| **2b — Demo zips** | Download 8 demo zips (4 participants × D65/F12) from GitHub; set `ZIP_PATH` |
| **3 — Inference** | Run best stack on `ZIP_PATH`; print forehead Lab, ΔE vs FitSkin forehead, ROI plots |
| **4 — Compare** | Same zip with frozen 5500 K baseline vs best stack (side-by-side Lab) |
| **5 — Batch** | *(Optional)* Run inference on every zip found in Cell 2b |
| **6 — Summary** | ΔE₀₀ table + bar chart for all 4 participants (D65 and F12) |

### How to run
1. **Runtime → GPU** optional (FairFace runs fine on CPU)
2. Run **Cell 1 (Setup) first**, then 1b → 2b → 3 → 6 (minimum path with torch plot + cohort summary)
3. Cells 2a, 4, and 5 are optional

Local CLI equivalent:
```bash
python scripts/run_d65_fairface7_roi.py \
  --zip path/to/AnjanaF12B1Torch.zip \
  --cat-mode hybrid_deploy \
  --multi-lab-corrector calibration/multi_illuminant_lab_affine/multi_illuminant_lab_affine.json \
  --roi forehead
```

> Torch SPD: uses `Torch_meas/` on Drive if present; otherwise falls back to flash SPD in `tier3_affine/iphone_calibration_bundle.json` (~4923 K).


## Cell 1 — Setup

**Run this cell first.** Installs packages, clones/pulls the Fitskin repo, adds it to `sys.path`, downloads FairFace-7 weights, and sets `REPO`, `CAL_DIR`, `TORCH_DIR`, etc.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — SETUP
# Clone repo, mount Drive, load calibration + FairFace-7 weights.
# ══════════════════════════════════════════════════════════════════════════════
!pip install -q rawpy opencv-python-headless numpy matplotlib gdown pandas

try:
    import torch, torchvision  # noqa: F401
except ImportError:
    !pip install -q torch torchvision

import json, sys, zipfile
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    pass

REPO_URL = "https://github.com/RooneyEmily/Fitskin.git"
if Path("Fitskin").is_dir():
    !cd Fitskin && git pull --ff-only 2>/dev/null || true
    REPO = Path("Fitskin").resolve()
elif (Path.cwd() / "pipeline" / "d65_fairface7_roi.py").is_file():
    REPO = Path.cwd().resolve()
else:
    !git clone -q {REPO_URL}
    REPO = Path("Fitskin").resolve()

if IN_COLAB and not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

sys.path = [str(REPO)] + [p for p in sys.path if Path(p).resolve() != REPO]

# Ensure best-stack calibration is present (may not be on remote git tip yet)
MULTI_LAB = REPO / "calibration" / "multi_illuminant_lab_affine" / "multi_illuminant_lab_affine.json"
PIPE = REPO / "pipeline" / "d65_fairface7_roi.py"
ASSET_ZIP = REPO / "colab_assets" / "ringlight_best_stack.zip"
SKIN_ROI = REPO / "pipeline" / "skin_roi.py"
need_assets = (
    (not MULTI_LAB.is_file())
    or (not PIPE.is_file())
    or (not SKIN_ROI.is_file())
)
if need_assets and ASSET_ZIP.is_file():
    print("Extracting colab_assets/ringlight_best_stack.zip …")
    with zipfile.ZipFile(ASSET_ZIP) as zf:
        zf.extractall(REPO)
elif need_assets:
    print("WARN: missing pipeline files — push colab_assets/ringlight_best_stack.zip to repo")

# Forehead ROI + best-stack pipeline (required for Cells 3+)
from pipeline.skin_roi import apple_face_skin_roi_mask  # noqa: E402

CAL_DIR = REPO / "calibration" / "tier3_affine"
FAIRFACE_DIR = REPO / "calibration" / "fairface"
FAIRFACE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = Path("/content/ringlight_best_stack_runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TORCH_DIR = Path("/content/drive/MyDrive/Torch_meas")  # optional; bundle fallback if missing

FF7 = FAIRFACE_DIR / "res34_fair_align_multi_7_20190809.pt"
if not FF7.is_file():
    print("Downloading FairFace-7 weights (~82 MB)…")
    !gdown 11y0Wi3YQf21a_VcspUV4FwqzhMcfaVAB -O "{FF7}"
assert FF7.is_file(), "FairFace weights missing"

import torch
from pipeline.d65_fairface7_roi import D65FairFace7ROIPipeline, write_result_json

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("REPO:", REPO)
print("tier3 affine:", (CAL_DIR / "camera_rgb_to_xyz_affine.npy").is_file())
print("multi-lab corrector:", MULTI_LAB.is_file())
print("Setup OK.")

## Cell 1b — Torch flash SPD

The Lu illuminant estimator uses the **measured iPhone torch flash** spectrum (~4923 K) as the flash chromaticity prior. This plot shows the SPD loaded for `hybrid_deploy` (from MK350 `Torch_meas/` on Drive, or from `tier3_affine/iphone_calibration_bundle.json` in the repo).

Same SPD + flash RGB prior are loaded inside **`hybrid_deploy`** inference (Lu 2006 ambient CCT from no-flash/flash pair), not just for this plot.


In [ ]:
# CELL 1b — Torch flash SPD (used by Lu + hybrid_deploy CAT)
import json
import matplotlib.pyplot as plt
import numpy as np
from pipeline.illuminant_estimation import load_torch_prior, load_torch_prior_from_cal_bundle

torch_source = "tier3 bundle"
try:
    td = TORCH_DIR if TORCH_DIR.is_dir() else None
    if td is not None:
        torch_prior = load_torch_prior(td)
        torch_source = str(td)
    else:
        raise FileNotFoundError
except Exception:
    torch_prior = load_torch_prior_from_cal_bundle(CAL_DIR)
    torch_source = f"{CAL_DIR}/iphone_calibration_bundle.json"

wl = np.asarray(torch_prior.wavelengths_nm, dtype=float)
spd = np.asarray(torch_prior.mean_spd, dtype=float)
spd_norm = spd / max(np.nanmax(spd), 1e-12)
rgb = np.asarray(torch_prior.flash_rgb)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(wl, spd_norm, color="#d84a2b", lw=2)
axes[0].set_xlabel("Wavelength (nm)")
axes[0].set_ylabel("Normalized SPD")
axes[0].set_title(f"iPhone torch flash SPD\nCCT ≈ {torch_prior.torch_cct_k:.0f} K")
axes[0].grid(True, alpha=0.3)

axes[1].bar(["R", "G", "B"], rgb, color=["#e74c3c", "#27ae60", "#3498db"], alpha=0.85)
axes[1].set_ylabel("Linear RGB (unit norm)")
axes[1].set_title("Torch flash chromaticity prior\n(flash_rgb in Lu 2006)")
axes[1].set_ylim(0, max(rgb) * 1.15)

plt.suptitle(f"Loaded from: {torch_source}", fontsize=10, y=1.02)
plt.tight_layout()
plt.show()
print(f"torch_cct_k={torch_prior.torch_cct_k:.1f}  flash_rgb={rgb.round(3).tolist()}")


## Cells 2a / 2b — Get a torch zip

**Cell 2b (default):** downloads **8 demo zips** (Anjana, Lihn, Parker, Woojae × D65/F12) from GitHub and sets `ZIP_PATH`.

**Cell 2a (optional):** upload your own `*Torch.zip` instead.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2a — OPTIONAL upload
# Set DO_UPLOAD=True, click Choose Files, pick one *Torch.zip (~20 MB).
# ══════════════════════════════════════════════════════════════════════════════
UPLOAD_DIR = Path("/content/uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

DO_UPLOAD = False  # True = pick one *Torch.zip in the widget below
print("Set DO_UPLOAD=True then click Choose Files below.")
if DO_UPLOAD and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        (UPLOAD_DIR / name).write_bytes(data)
        print("saved", UPLOAD_DIR / name)
else:
    print("Upload skipped — use Cell 2b Drive path or set DO_UPLOAD=True")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2b — Demo zips from GitHub
# Downloads 8 demo zips (4 participants × D65/F12) from GitHub; sets ZIP_PATH for Cells 3–5. (no Drive / upload needed)
# ══════════════════════════════════════════════════════════════════════════════
import json
import urllib.request

from pipeline.illuminant_estimation import infer_illuminant_label

DEMO_PERSON = "Anjana"       # Anjana | Lihn | Parker | Woojae
PREFER_ILLUMINANT = "F12"    # "F12" or "D65"

DEMO_DIR = REPO / "data" / "ring_light" / "demo_zips"
MANIFEST_PATH = REPO / "data" / "ring_light" / "demo_manifest.json"
GITHUB_RAW = "https://github.com/RooneyEmily/Fitskin/raw/main/data/ring_light/demo_zips"

# TORCH_DIR set in Cell 1 (optional Drive override here if needed)
DEMO_DIR.mkdir(parents=True, exist_ok=True)

if MANIFEST_PATH.is_file():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
else:
    manifest = {
        "demos": [
            {"file": "AnjanaF12B1Torch.zip", "person": "Anjana", "illuminant": "F12"},
            {"file": "Anjana-D65-C3Torch.zip", "person": "Anjana", "illuminant": "D65"},
            {"file": "Lihn-F12-B1Torch.zip", "person": "Lihn", "illuminant": "F12"},
            {"file": "LihnD65-C1Torch.zip", "person": "Lihn", "illuminant": "D65"},
        ]
    }

for d in manifest.get("demos", []):
    name = d["file"]
    dest = DEMO_DIR / name
    if dest.is_file() and dest.stat().st_size > 1_000_000:
        print("have", name)
        continue
    url = f"{GITHUB_RAW}/{name}"
    print(f"Downloading {name} (~20 MB) …")
    urllib.request.urlretrieve(url, dest)
    print("  ->", dest, f"({dest.stat().st_size/1e6:.1f} MB)")

candidates = sorted(DEMO_DIR.glob("*.zip"))
# optional: also pick up manual uploads
if UPLOAD_DIR.is_dir():
    candidates = sorted(set(candidates) | set(UPLOAD_DIR.glob("*.zip")), key=lambda p: p.name)

print(f"\nDemo zips ready ({len(candidates)}):")
for p in candidates:
    print(f"  [{infer_illuminant_label(p) or '?'}] {p.name}")

def _person_match(path: Path, person: str) -> bool:
    s = path.name.lower()
    aliases = {"anjana": ("anjana",), "lihn": ("lihn", "linh"), "parker": ("parker",), "woojae": ("woojae", "wooj")}
    keys = aliases.get(person.lower(), (person.lower(),))
    return any(k in s for k in keys)

person_pool = [p for p in candidates if _person_match(p, DEMO_PERSON)]
if not person_pool:
    raise RuntimeError(f"No demo zip for {DEMO_PERSON!r} in {DEMO_DIR}")

ill = PREFER_ILLUMINANT.upper()
ZIP_PATH = next((p for p in person_pool if infer_illuminant_label(p) == ill), person_pool[0])

print(f"\nSelected ({DEMO_PERSON}, {PREFER_ILLUMINANT}):", ZIP_PATH)
print("Inferred illuminant:", infer_illuminant_label(ZIP_PATH))


## Cell 3 — Inference + plots

Loads the production pipeline (`hybrid_deploy` + multi-lab corrector, **forehead ROI**), runs on `ZIP_PATH` from Cell 2b, writes JSON to `/content/ringlight_best_stack_runs/`, and shows forehead mask / FairFace / Lab swatch figures.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Inference + visuals
# hybrid_deploy + multi-lab on ZIP_PATH; optional ΔE vs FitSkin forehead.
# ══════════════════════════════════════════════════════════════════════════════
import cv2
import matplotlib.pyplot as plt
import numpy as np
from pipeline.skin_roi import apple_face_skin_roi_mask, load_apple_landmarks
from scripts.evaluate_pansor20_chartfree_d65 import (
    extract_zip,
    linear_rgb_to_preview_bgr,
    load_dng_linear,
)
from models.fairface_race import face_rgb_crop_from_landmarks

CAT_MODE = "hybrid_deploy"
SAMPLING = "fairface7"  # "off" = trimmed mean only
ROI = "forehead"  # FitSkin scan site

pipe = D65FairFace7ROIPipeline.from_defaults(
    cal_dir=CAL_DIR,
    fairface_dir=FAIRFACE_DIR,
    cat_mode=CAT_MODE,
    torch_dir=TORCH_DIR if TORCH_DIR.is_dir() else None,
    multi_lab_affine=MULTI_LAB,
    half_size=True,
    sampling=SAMPLING,
    roi=ROI,
)
print(f"Pipeline: cat_mode={CAT_MODE}  roi={ROI}  multi_lab={MULTI_LAB.name}  sampling={SAMPLING}")

result = pipe.run_zip(ZIP_PATH)
out_json = OUT_DIR / f"{ZIP_PATH.stem}.json"
write_result_json(result, out_json)

# FitSkin forehead reference (Booth Lighting) — same ROI as pipeline
_ref = None
if MANIFEST_PATH.is_file():
    _manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    for _d in _manifest.get("demos", []):
        if _d.get("file") == ZIP_PATH.name:
            _ref = _d.get("fitskin_forehead")
            break
if _ref:
    from delta_e_2000 import delta_e_2000
    import numpy as np
    ref_lab = np.array([_ref["L"], _ref["a"], _ref["b"]])
    pred_lab = np.array([result["L"], result["a"], result["b"]])
    de = float(delta_e_2000(pred_lab, ref_lab))
    print(
        f"\nFitSkin forehead (Booth): L*={_ref['L']:.2f} a*={_ref['a']:.2f} b*={_ref['b']:.2f}"
        f"  |  ΔE00={de:.2f}"
    )

_cat = result.get("cat_cct")
_cat_s = f"{float(_cat):.0f} K" if _cat is not None else "?"
_lu = result.get("lu_cct_k")
_lu_s = f"{float(_lu):.0f} K" if _lu is not None else "n/a"
print(
    f"\nLab = ({result['L']:.2f}, {result['a']:.2f}, {result['b']:.2f})\n"
    f"illuminant = {result.get('illuminant_label')}  "
    f"CAT CCT = {_cat_s}  Lu CCT = {_lu_s}\n"
    f"lab_corrector = {result.get('lab_corrector')}  "
    f"FairFace = {result.get('fairface_label')} → {result.get('predicted_ethnicity')} "
    f"(conf={result.get('fairface_confidence'):.2f})\n"
    f"roi={result.get('roi')}  n_roi={result.get('n_roi')}  flash_scale = {result.get('flash_scale'):.3f}"
)
ef = result.get("exposure_flags") or {}
if ef.get("out_of_band"):
    print("⚠ exposure_flags:", ef)
else:
    print("exposure_flags: OK")
if result.get("lu_cct_k") is not None:
    print(
        f"\nTorch SPD used in hybrid_deploy: Lu ambient CCT ≈ {float(result['lu_cct_k']):.0f} K "
        f"→ Bradford W_src (see cat_cct={result.get('cat_cct')})"
    )
print("Wrote", out_json)

# ── visuals ─────────────────────────────────────────────────────────────────
work = OUT_DIR / "_viz"
nf, fl, lm_path = extract_zip(ZIP_PATH, work / ZIP_PATH.stem)
A0 = load_dng_linear(nf, half_size=True, use_camera_wb=False)
lm = load_apple_landmarks(lm_path)
forehead = apple_face_skin_roi_mask(lm, A0.shape[0], A0.shape[1], roi="forehead", linear_rgb=A0)
preview = linear_rgb_to_preview_bgr(A0)
overlay = preview.copy()
overlay[forehead > 0] = (0.55 * overlay[forehead > 0] + 0.45 * np.array([0, 220, 80])).astype(np.uint8)
face_rgb = face_rgb_crop_from_landmarks(preview, lm, padding=0.35)

def lab_to_srgb_u8(L, a, b):
    fy = (L + 16.0) / 116.0
    fx, fz = fy + a / 500.0, fy - b / 200.0
    eps, kappa = 216 / 24389, 24389 / 27
    def finv(t):
        return t**3 if t**3 > eps else (116 * t - 16) / kappa
    X, Y, Z = 0.95047 * finv(fx), finv(fy), 1.08883 * finv(fz)
    M = np.array([[3.2406, -1.5372, -0.4986], [-0.9689, 1.8758, 0.0415], [0.0557, -0.2040, 1.0570]])
    rgb = M @ np.array([X, Y, Z])
    lin2s = lambda u: 12.92 * u if u <= 0.0031308 else 1.055 * (max(u, 0) ** (1 / 2.4)) - 0.055
    return (np.clip([lin2s(float(c)) for c in rgb], 0, 1) * 255).astype(np.uint8)

swatch = np.full((180, 180, 3), lab_to_srgb_u8(result["L"], result["a"], result["b"]), dtype=np.uint8)

fig, ax = plt.subplots(1, 4, figsize=(14, 3.6))
ax[0].imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB)); ax[0].set_title("No-flash"); ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); ax[1].set_title(f"Forehead ROI (n={result['n_roi']})"); ax[1].axis("off")
ax[2].imshow(face_rgb); ax[2].set_title(f"FairFace-7\n{result.get('fairface_label')} → {result.get('predicted_ethnicity')}"); ax[2].axis("off")
ax[3].imshow(swatch); ax[3].set_title(f"Forehead Lab\n({result['L']:.1f}, {result['a']:.1f}, {result['b']:.1f})"); ax[3].axis("off")
plt.suptitle(f"{ZIP_PATH.name}  ·  hybrid_deploy + multi-lab  ·  {result.get('illuminant_label')}", fontsize=11)
plt.tight_layout()
plt.show()

## Cell 4 — Baseline comparison

Runs the **same zip** twice: frozen 5500 K CAT (indoor baseline) vs the best stack, so you can see how much illuminant routing and the Lab corrector shift L*a*b*.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Compare arms
# Same zip: frozen_5500 baseline vs hybrid_deploy + multi-lab.
# ══════════════════════════════════════════════════════════════════════════════
pipe_frozen = D65FairFace7ROIPipeline.from_defaults(
    cal_dir=CAL_DIR,
    fairface_dir=FAIRFACE_DIR,
    cat_mode="frozen_5500",
    half_size=True,
    sampling=SAMPLING,
    roi=ROI,
)
r_frozen = pipe_frozen.run_zip(ZIP_PATH)

print(f"{'Arm':<22} {'L*':>7} {'a*':>7} {'b*':>7} {'CAT K':>8} {'corrector':<12}")
print("-" * 70)
for label, r in [("frozen_5500 (baseline)", r_frozen), ("best stack", result)]:
    print(
        f"{label:<22} {r['L']:7.2f} {r['a']:7.2f} {r['b']:7.2f} "
        f"{float(r.get('cat_cct') or 0):8.0f} {str(r.get('lab_corrector') or '-'):<12}"
    )
dL, da, db = result["L"] - r_frozen["L"], result["a"] - r_frozen["a"], result["b"] - r_frozen["b"]
print(f"\nΔLab (best − frozen): ΔL*={dL:+.2f}  Δa*={da:+.2f}  Δb*={db:+.2f}")

## Cell 5 — Batch (optional)

Set `RUN_BATCH = True` to process every demo zip and write a summary JSON. Reuses the pipeline loaded in Cell 3.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Optional batch
# Set RUN_BATCH=True to run every demo zip and write batch_summary.json. (reuse `pipe` from Cell 3)
# ══════════════════════════════════════════════════════════════════════════════
RUN_BATCH = False  # set True to process all zips found in Cell 2b

if RUN_BATCH:
    summary = []
    for i, zp in enumerate(candidates, 1):
        try:
            r = pipe.run_zip(zp)
        except Exception as exc:
            print(f"[{i:02d}/{len(candidates)}] FAIL {zp.name}: {exc}")
            summary.append({"zip": str(zp), "error": str(exc)})
            continue
        out = OUT_DIR / f"{zp.stem}.json"
        write_result_json(r, out)
        print(
            f"[{i:02d}/{len(candidates)}] {zp.name:36s}  "
            f"Lab=({r['L']:.1f},{r['a']:.1f},{r['b']:.1f})  "
            f"ill={r.get('illuminant_label')}  cat={r.get('cat_cct'):.0f}K  "
            f"FF={r.get('fairface_label')}→{r.get('predicted_ethnicity')}"
        )
        summary.append({
            "zip": str(zp), "out": str(out),
            "L": r["L"], "a": r["a"], "b": r["b"],
            "illuminant": r.get("illuminant_label"),
            "cat_cct": r.get("cat_cct"),
            "lab_corrector": r.get("lab_corrector"),
            "fairface_label": r.get("fairface_label"),
            "predicted_ethnicity": r.get("predicted_ethnicity"),
        })
    batch_summary = OUT_DIR / "batch_summary.json"
    batch_summary.write_text(json.dumps({"n": len(summary), "trials": summary}, indent=2) + "\n")
    print("Wrote", batch_summary)
else:
    print("Batch skipped (set RUN_BATCH=True).")

## Cell 6 — Cohort summary (4 participants × D65 / F12)

Runs the **best stack** (forehead ROI) on every demo zip in the manifest, compares predicted Lab to FitSkin **forehead** reference, and reports ΔE₀₀ by participant and ring illuminant. Also shows frozen 5500 K baseline ΔE₀₀ for context.


In [ ]:
# CELL 6 — Summary ΔE00 for all demo participants
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from delta_e_2000 import delta_e_2000

if "pipe" not in globals():
    pipe = D65FairFace7ROIPipeline.from_defaults(
        cal_dir=CAL_DIR,
        fairface_dir=FAIRFACE_DIR,
        cat_mode="hybrid_deploy",
        torch_dir=TORCH_DIR if TORCH_DIR.is_dir() else None,
        multi_lab_affine=MULTI_LAB,
        half_size=True,
        sampling="fairface7",
        roi="forehead",
    )
if "pipe_frozen" not in globals():
    pipe_frozen = D65FairFace7ROIPipeline.from_defaults(
        cal_dir=CAL_DIR,
        fairface_dir=FAIRFACE_DIR,
        cat_mode="frozen_5500",
        half_size=True,
        sampling="fairface7",
        roi="forehead",
    )

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
rows = []
for d in manifest.get("demos", []):
    zp = DEMO_DIR / d["file"]
    if not zp.is_file():
        print("SKIP missing", d["file"])
        continue
    ref = d.get("fitskin_forehead") or {}
    ref_lab = np.array([ref["L"], ref["a"], ref["b"]], dtype=float)
    r_best = pipe.run_zip(zp)
    r_frozen = pipe_frozen.run_zip(zp)
    pred_best = np.array([r_best["L"], r_best["a"], r_best["b"]])
    pred_frozen = np.array([r_frozen["L"], r_frozen["a"], r_frozen["b"]])
    rows.append({
        "person": d.get("person", ""),
        "illuminant": d.get("illuminant", ""),
        "zip": d["file"],
        "pred_L": r_best["L"], "pred_a": r_best["a"], "pred_b": r_best["b"],
        "ref_L": ref.get("L"), "ref_a": ref.get("a"), "ref_b": ref.get("b"),
        "de00_best": float(delta_e_2000(pred_best, ref_lab)),
        "de00_frozen": float(delta_e_2000(pred_frozen, ref_lab)),
        "cat_cct": r_best.get("cat_cct"),
        "lu_cct_k": r_best.get("lu_cct_k"),
        "lab_corrector": r_best.get("lab_corrector"),
    })
    print(f"{d['person']:7} {d['illuminant']:3}  best ΔE={rows[-1]['de00_best']:.2f}  frozen ΔE={rows[-1]['de00_frozen']:.2f}")

df = pd.DataFrame(rows)
summary_path = OUT_DIR / "demo_cohort_summary.csv"
df.to_csv(summary_path, index=False)
print("\nWrote", summary_path)

pivot_best = df.pivot(index="person", columns="illuminant", values="de00_best").reindex(columns=["D65", "F12"])
pivot_frozen = df.pivot(index="person", columns="illuminant", values="de00_frozen").reindex(columns=["D65", "F12"])

print("\nΔE₀₀ — best stack (hybrid_deploy + multi-lab), forehead vs FitSkin forehead:")
display(pivot_best.round(2))
print(f"Mean all trials: {df['de00_best'].mean():.2f}  |  D65 mean: {df.loc[df.illuminant=='D65','de00_best'].mean():.2f}  |  F12 mean: {df.loc[df.illuminant=='F12','de00_best'].mean():.2f}")

print("\nΔE₀₀ — frozen 5500 K baseline:")
display(pivot_frozen.round(2))
print(f"Mean all trials: {df['de00_frozen'].mean():.2f}  |  D65 mean: {df.loc[df.illuminant=='D65','de00_frozen'].mean():.2f}  |  F12 mean: {df.loc[df.illuminant=='F12','de00_frozen'].mean():.2f}")

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(df))
w = 0.35
ax.bar(x - w/2, df["de00_frozen"], w, label="frozen_5500", color="#4C72B0", alpha=0.85)
ax.bar(x + w/2, df["de00_best"], w, label="hybrid_deploy + multi-lab", color="#C44E52", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([f"{r.person}\n{r.illuminant}" for r in df.itertuples()], fontsize=9)
ax.set_ylabel("ΔE₀₀ vs FitSkin forehead")
ax.set_title("Demo cohort (n=8): forehead pipeline vs FitSkin reference")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## Reference — pinned ring-light eval (n=84)

Legacy table used **cheek** ROI vs FitSkin **forehead** reference. After this update, re-run:

```bash
python scripts/evaluate_ringlight_torch_illuminant.py --roi forehead
```

| Arm | All mean ΔE₀₀ | D65 ring | F12 ring |
|---|---:|---:|---:|
| frozen_5500 | 9.75 | 6.46 | 13.37 |
| hybrid_deploy | 9.28 | 6.81 | 12.00 |
| **hybrid_deploy + multi-lab** | **~8.5** | **~8.0** | **~9.1** |

**Ring CC-supervised affines (separate experiment):** in-frame ColorChecker training reaches ~8 ΔE on chart patches; skin ROI on the same captures remains much higher, so the deployed stack keeps indoor `tier3_affine` on skin.

Rebuild the optional Colab asset zip locally:
```bash
python scripts/build_ringlight_colab_assets.py
```
